# Descriptive Statistics with Banking Operations

**Dataset:** `banking_operations.csv`  
**Tools:** pandas, NumPy, Matplotlib and topic-specific statistical/ML functions  

This notebook explains the concept in simple terms and connects every calculation to banking operations.

## 1. What descriptive statistics does

Descriptive statistics summarizes the dataset we already have. It helps us understand a typical transaction, variation in amounts, unusual values, category frequencies and operational patterns. It does **not** establish cause-and-effect.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

df = pd.read_csv("banking_operations.csv")
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

print("Dataset shape:", df.shape)
display(df.head())

## 2. Data-quality overview

Before calculating statistics, check column types and missing values. Missing categories are retained as `Unknown` so that transactions are not silently removed.

In [ ]:
display(df.dtypes.to_frame("Data type"))
display(df.isna().sum().to_frame("Missing values"))

df["Channel"] = df["Channel"].fillna("Unknown")
df["Branch_City"] = df["Branch_City"].fillna("Unknown")

## 3. Mean, median and mode

- **Mean:** arithmetic average; sensitive to unusually large transactions.
- **Median:** middle value; more resistant to extreme values.
- **Mode:** most frequent value; useful for both numeric and categorical data.

In [ ]:
centre = pd.Series({
    "Mean Amount": df["Amount"].mean(),
    "Median Amount": df["Amount"].median(),
    "Mode Amount": df["Amount"].mode().iloc[0],
    "Most Common Status": df["Status"].mode().iloc[0],
    "Most Common Channel": df["Channel"].mode().iloc[0]
})
display(centre.to_frame("Result"))

## 4. Spread: range, variance, standard deviation and coefficient of variation

Variance uses squared units. Standard deviation returns to the original unit. The coefficient of variation expresses standard deviation relative to the mean.

In [ ]:
amount_mean = df["Amount"].mean()
amount_variance = df["Amount"].var(ddof=1)
amount_std = df["Amount"].std(ddof=1)
amount_range = df["Amount"].max() - df["Amount"].min()
amount_cv = amount_std / amount_mean * 100

spread = pd.Series({
    "Minimum": df["Amount"].min(),
    "Maximum": df["Amount"].max(),
    "Range": amount_range,
    "Sample Variance": amount_variance,
    "Sample Standard Deviation": amount_std,
    "Coefficient of Variation (%)": amount_cv
})
display(spread.to_frame("Amount"))

## 5. Quartiles, percentiles and IQR outliers

Q1, Q2 and Q3 divide ordered values into four parts. The interquartile range (IQR) measures the middle 50% and helps flag unusual values for review.

In [ ]:
q1 = df["Amount"].quantile(0.25)
q2 = df["Amount"].quantile(0.50)
q3 = df["Amount"].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

quartiles = pd.Series({"Q1": q1, "Q2 / Median": q2, "Q3": q3, "IQR": iqr,
                       "Lower Review Limit": lower_limit, "Upper Review Limit": upper_limit})
display(quartiles.to_frame("Value"))

outliers = df[(df["Amount"] < lower_limit) | (df["Amount"] > upper_limit)]
display(outliers[["Transaction_ID", "Amount", "Transaction_Type", "Status"]])

## 6. Grouped banking summaries

Grouped statistics show whether transaction behaviour differs across channels and statuses.

In [ ]:
channel_summary = df.groupby("Channel", dropna=False).agg(
    Transactions=("Transaction_ID", "count"),
    Average_Amount=("Amount", "mean"),
    Median_Amount=("Amount", "median"),
    Amount_Std_Dev=("Amount", "std"),
    Total_Amount=("Amount", "sum")
).sort_values("Total_Amount", ascending=False)
display(channel_summary)

In [ ]:
df["Amount"].plot(kind="hist", bins=10, edgecolor="black", color="#2F80ED", figsize=(8, 4))
plt.axvline(df["Amount"].mean(), color="red", linestyle="--", label="Mean")
plt.axvline(df["Amount"].median(), color="green", linestyle="-.", label="Median")
plt.title("Distribution of Banking Transaction Amounts")
plt.xlabel("Amount")
plt.legend()
plt.show()

## Banking interpretation

Use the mean and median together to describe a typical transaction. Use standard deviation and IQR to understand volatility. Review outliers rather than automatically deleting them: a large transaction can be legitimate, a data error or an operational-risk signal.